# Step 5: 从 AOP 做经典重建,看灰度图

这一节直接使用经典重建 API：

- `TD_integration`
- `SD_integration`
- `poisson_blending`
- `SD2XY`

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'tianmoucv').exists() and (candidate / 'tianmoucv_example').exists():
        repo_root = candidate
        break
if repo_root is None:
    raise RuntimeError('找不到 tianmoucv 仓库根目录，请从仓库内打开本 Notebook。')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
import torch


In [ ]:
import torch.nn.functional as F
from tianmoucv.data import TianmoucDataReader
from tianmoucv.isp import SD2XY
from tianmoucv.proc.reconstruct import SD_integration, TD_integration, poisson_blending
from tianmoucv.utils import ensure_tutorial_dataset

info = ensure_tutorial_dataset()
dataset = TianmoucDataReader(str(info.sample_dir), N=1, print_info=False)
sample = dataset[4]
rawdiff = sample['rawDiff'].float() / 128.0
F0 = sample['F0']
F1 = sample['F1']

In [ ]:
t_index = 12
hdr_td = TD_integration(rawdiff, F0, F1, t_index, threshGate=0.0)

sd = rawdiff[1:, t_index].permute(1, 2, 0)
ix, iy = SD2XY(sd)
gray_sd = SD_integration(ix, iy, method='multi_path', anchor=F0, return_tensor=True)
ix_full = F.interpolate(ix.unsqueeze(0).unsqueeze(0), size=(320, 640), mode='bilinear', align_corners=False).squeeze()
iy_full = F.interpolate(iy.unsqueeze(0).unsqueeze(0), size=(320, 640), mode='bilinear', align_corners=False).squeeze()
hdr_poisson = poisson_blending(ix_full, iy_full, iteration=20)

In [ ]:
plt.figure(figsize=(14, 8))
plt.subplot(2, 2, 1)
plt.imshow(F0.numpy())
plt.title('F0')
plt.axis('off')

plt.subplot(2, 2, 2)
plt.imshow(hdr_td.numpy(), cmap='gray')
plt.title('TD_integration')
plt.axis('off')

plt.subplot(2, 2, 3)
plt.imshow(gray_sd.numpy(), cmap='gray')
plt.title('SD_integration(Ix, Iy)')
plt.axis('off')

plt.subplot(2, 2, 4)
plt.imshow(hdr_poisson.numpy(),cmap='gray')
plt.title('poisson_blending')
plt.axis('off')
plt.tight_layout()
plt.show()